<a href="https://colab.research.google.com/github/kjfcvx12/Colab/blob/main/04-28%2002.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 시계열
# optuna : 하이퍼파라미터 최적화 자동화해주는 프레임워크
# 모델성능을 최고로 끌어올리기 위한 최적의 설정값 조합을 알고리즘이 알아서 빠르게 찾아줌
# 가지치기/TPE
!pip install optuna-integration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

In [ ]:
from sklearn.preprocessing import StandardScaler

try:
  from optuna_integration import PyTorchPruningCallback
except ImportError:
  try:
    from optuna.integration import PyTorchPruningCallback
  except ImportError:
    PyTorchPrunginCallback=None

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
# 금융데이터 불러올수 있는 라이브러리
!pip install -U finance-datareader

In [ ]:
import FinanceDataReader as fdr

df=fdr.DataReader('000660', start='2016-04-28')
display(df.head())
display(df.tail())
# 결측지 있는지 확인
df.info()

In [ ]:
df.isna().sum()

In [ ]:
# 금융차트 그리는데 특화됨
!pip install mplfinance

In [ ]:
# 데이터 정하고 - 데이터 정제 - 데이터 분석(EDAI : Exploratory Data Anlysis) - 성능평가

In [ ]:
from pygments import style
import mplfinance as mpf

#최근 2달 데이터 추출
data_sub=df.tail(60)

#스타일 커스터마이즈
mc=mpf.make_marketcolors(up='red',down='blue',edge='inherit',wick='black',volume='inherit')
s=mpf.make_mpf_style(marketcolors=mc, gridstyle='--')

mpf.plot(data_sub, type='candle',style=s,
         title='SK Hynix Chart',
         ylabel='Price',
         volume=True,
         #moving average value
         #5일선(5일동안 종가 평균), 20일선(20일동안 종가 평균)
         mav=(5,20),
         figsize=(12,8))

plt.show()

In [ ]:
#연도만 추출해서 새 열 만듬
df['Year']=df.index.year

In [ ]:
# 10년간 하이닉스 주가 어떻게 변해왔는지 전체적 흐름
import seaborn as sns
sns.lineplot(data=df, x=df.index, y='Close')
plt.show()

In [ ]:
# pct_change() : percent change(백분율 변화)
# 수익률
df['Return']=df['Close'].pct_change()

In [ ]:
# 거래량
df['Volume_Change']=df['Volume'].pct_change()
df

In [ ]:
# 수익률과 거래량의 상관계수
df[['Return', 'Volume_Change']].corr()

In [ ]:
import yfinance as yf
from tqdm.auto import tqdm

In [ ]:
from enum import auto
hynix_code='000660.KS'
df=yf.download(hynix_code, start='2016-04-28', auto_adjust=True)
df

In [ ]:
# df.columns 이 MultIndex (컬럼이 계층형으로 되어있어서) 인지 확인 후
# 단일컬럼으로 바꿈
if isinstance(df.columns, pd.MultiIndex):
  df.columns=df.columns.get_level_values(0)

In [ ]:
df.columns

In [ ]:
# 현재값 이전값 사이의 변화율 계산
# (현재값-이전값)/이전값
# 시간에 따라 우상향, 단위가 커지면 모델에 학습과 어렵다 -> % 변화율로 데이터 범위를 일정하게 유지해야 유리
df['Return']=df['Close'].pct_change()
features_df=df[['Close', 'High', 'Low', 'Open', 'Volume']].pct_change()

In [ ]:
# 결측치 처리
features_df.isna().sum()

In [ ]:
# 변화율(pct_change) 사용했기 때문에 무한대값이 있을 수 있음
# (100-0)/0 = 무한대를 NaN으로 대체
features_df.replace([np.inf, -np.inf], np.nan, inplace=True)
features_df.fillna(0, inplace=True)
target_return=df['Return'].replace([np.inf, -np.inf], np.nan).fillna(0)

In [ ]:
# 수치데이터들만 뽑음 .values
features=features_df.values
target=target_return.values

In [ ]:
train_size=int(len(features))
train_size

In [ ]:
# 80프로는 train, 20프로는 test
# 주가 데이터는 학습데이터와 테스트 데이터 나눌 때 shuffle하면 큰일남 => 시간 순서대로 잘라야 하니까
train_size=int(len(features)*0.8)
X_train, X_test=features[:train_size], features[train_size:]
y_train,y_test=target[:train_size], target[train_size:]

In [ ]:
# 스케일링 Standard
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [ ]:
# X: 피처들(모델의 입력이 될 피처들(시가, 중가, 고가, 저가, 거래량))
# y : 내일의 수읽률(맞춰야 하는 정답)
# window_size=20 : 모델이 한번에 참고할 과거 데이터의 기간
def create_seq(X,y,winodw_size=20):
  xs, ys=[], []
  # 전체데이터 끝-20일 뺀 지점까지만 반복
  for i in range(len(X)-winodw_size):
    xs.append(X[i:i+winodw_size]) # 총 20개 데이터 잘라냄
    ys.append(y[i+winodw_size]) # 위에서 자른 20일 데이터 바로 다음 날
  return np.array(xs), np.array(ys)

In [ ]:
X_train, y_train=create_seq(X_train_scaled,y_train,20)
X_test, y_test=create_seq(X_test_scaled,y_test,20)

In [ ]:
# X_train.shape : (전체 샘플수, window_size, feature수)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

In [ ]:
# 데이터를 1~20일 문제 정답

# 데이터셋/dataloader
class StockDataset(Dataset):
  def __init__(self, X, y):
    self.X=torch.tensor(X, dtype=torch.float32)
    # 맨 뒤 차원 추가 ->
    # 파이토치 [7.3, 3.1, 6.4]=>[[0.4],
    #                            [6.3],
    #                            [2.1]]
    self.y=torch.tensor(y, dtype=torch.float32).unsqueeze(-1) #(3,1)


  def __len__(self):
    return len(self.X)

  def __getitem__(self, i):
    return self.X[i], self.y[i]


In [ ]:
# 데이터 분할시 -> 셔플 절대금지(미래 데이터가 학습에 포함되는 데이터 누수 방지하기 위해)
# 학습 시 -> 셔플 권장(특정기간에 편향 안되고 다양한 패턴 골고루 학습)
# 평가 시 -> 셔플 비권장
train_loader=DataLoader(StockDataset(X_train, y_train), batch_size=32, shuffle=True)
test_loader=DataLoader(StockDataset(X_test, y_test), batch_size=32, shuffle=False)

In [ ]:
# 순환 신경망 모델 정의
class StockLSTM(nn.Module):
  def __init__(self, input_dim):
    super().__init__()
    # (배치, )
    # 64->32->1(수익률)
    self.lstm=nn.LSTM(input_dim, 64, num_layers=2, batch_first=True, dropout=0.2)
    self.fc=nn.Sequential(nn.Linear(64,32), nn.ReLU(), nn.Linear(32, 1))

  # 입력 : (32, 20, 5) => 출력 : 배치사이즈, sequence length, hidden size
  # 입력 x는 3차원 [배치사이즈, sequence length, input feature]
  def forward(self, x):
    # out -> 3차원[배치, 날짜(시퀀스), 특징]
    out,_=self.lstm(x)
    # [모든 배치에 대해 가장 마지막의 출력값만 뽑아낸다]
    # 데이터 전체를 다 가져와 : 20일치 날짜 중 마지막 것만 : 마지막에 들어있는 모든 정보(5개)
    return self.fc(out[:, -1, :])

In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
model=StockLSTM(features.shape[1]).to(device)
optimizer=optim.Adam(model.parameters(), lr=0.001)
mseloss=nn.MSELoss()

In [ ]:
# 32개씩 잘라서 섞어서 데이터 가져와서 2개 lstm로 최근 추세를 읽음
# 마지막 날 정보를 바탕으로 내일 수익률이 얼마일지 하나의 숫자로 찍어냄

In [ ]:
model.train()
for epoch in tqdm(range(300), desc="Training"):
    for b_x, b_y in train_loader:
        b_x, b_y = b_x.to(device), b_y.to(device)
        optimizer.zero_grad()
        loss = mseloss(model(b_x), b_y)
        loss.backward()
        optimizer.step()

In [ ]:
# 성능평가
def evaluate(loader, data_name):
  model.eval()
  t_loss=0

  with torch.no_grad():
    for b_x, b_y in loader:
        b_x, b_y = b_x.to(device), b_y.to(device)
        output=model(b_x)
        # 현재 그룹의 오차총합을 구해야 하니까 -> 32*평균오차
        t_loss+=mseloss(output, b_y).item()*b_x.size(0)
  # 전체 데이터의 오차 총합/실제 데이터 총 개수
  avg_loss=t_loss/len(loader.dataset)
  print(avg_loss)
  return avg_loss

In [ ]:
evaluate(train_loader, "Train Data")
print("="*30)
evaluate(test_loader, "Test Data")
# 학습데이터 mse ->  0.0000084986
# 테스트데이터 mse -> 0.0019275

In [ ]:
window_size=20
model.eval()
with torch.no_grad():
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
    test_preds_returns = model(X_test_tensor).cpu().numpy()

# 가격 복원
# base_prices : 예측하려는 날의 바로 전날 종가
base_prices = df['Close'].values[train_size + window_size - 1 : -1]
# 5%상승 => 1.05를 곱해야 내일 가격이 나오니까
predicted_prices = base_prices * (1 + test_preds_returns.flatten())
# 실제가격
actual_prices = df['Close'].values[train_size + window_size:]

In [ ]:
# train_size : 모델이 학습할 때 사용한 과거 데이터 양
# winodw_size : 20일이명 21일 째 가격을 예측하기 위해서
#               앞의 20일치 데이터가 필요하다
# 테스트 데이터의 첫번째 예측값
# => train_size로부터

In [ ]:
# base_prices[0] : 월요일 종가
# actual_prices[0] : 화요일 종가

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(actual_prices, label='Actual Price', color='blue')
plt.plot(predicted_prices, label='Predicted Price', color='crimson', linestyle='--')
plt.title('SK Hynix Stodck Predtion')

In [ ]:
# 내일 종가 계산
# 정규화된 테스트데이터[-20] -> 가장 가까운 과거 20일 동안
# 내일 예측하려면 바로 20일간의 흐름을 알아야함
last_seq=X_test_scaled[-window_size:]
# 텐서로 변환
last_seq_tensor=torch.tensor(last_seq, dtype=torch.float32)
last_seq_tensor=last_seq_tensor.unsqueeze(0).to(device)
last_seq_tensor.shape

In [ ]:
with torch.no_grad():
  tomorrow_pred=model(last_seq_tensor).item()

# 오늘의 실제종가
# 결측치 있다면 앞선 데이터로 채우겠다.
# iloc[-1] : 데이터프레임의 맨 마지막 행 (가장 최근 가격)
current_price=df['Close'].ffill().iloc[-1]
tomorrow_price=current_price *(1+tomorrow_pred)

print(f"현재가 : {current_price : ,.0f}원")
print(f"내일 예상 종가 : {tomorrow_price : ,.0f}원")